# Meet Hermes: Your First AI **Agent** 🛠️

**NAIRR Workshop 2026 — installing the [Hermes Agent](https://github.com/NousResearch/hermes-agent) and pointing it at a local model on a CPU instance**

In notebook 04 we ran *engines* (Ollama, llama.cpp, Transformers) — programs that **answer a question**. This notebook is about something different: an **agent**.

> **Engine vs. agent — the one idea to take away**
> - An **engine** takes a prompt and returns text. That's it.
> - An **agent** uses a model as a *brain* to **take actions** — it can call tools, run commands, remember things across conversations, and even build new skills for itself. The model decides *what to do*, not just *what to say*.

**Hermes** (by Nous Research) is an open-source agent that does exactly this — "the agent that grows with you." We'll **install it**, **point it at the same local Ollama model** from notebook 04 (so it runs free, on CPU, with no API key), and then **chat with it in the terminal**.

## How this notebook works (read this first)

Hermes is a **terminal application** — its real interface is a full-screen chat you run with the `hermes` command. That kind of interface can't run *inside* a Jupyter cell, so we split the work:

1. **In this notebook (the cells below):** install Hermes and configure it to use your local Ollama model. These are quick, non-interactive setup steps.
2. **In the Terminal (the last step):** actually *talk* to the agent and watch it work.

> ⚠️ **Prerequisite:** this builds on **notebook 04** — it needs **Ollama** installed and running. The first cell below will set that up for you if it isn't already, so you can run this notebook on its own too.

## 1. Make sure a local model is ready (Ollama)

Hermes needs a "brain" — a model to think with. Instead of a paid cloud API, we point it at **Ollama** running right here on the instance. This cell makes sure Ollama is installed, running, and has a small model pulled.

> **The model:** `qwen2.5:3b` is small enough for a CPU instance but capable enough to *sometimes* use tools — which is what makes an agent interesting. On a tight disk you can change it to `qwen2.5:0.5b` (faster, weaker at tools).

In [ ]:
import subprocess, os, time, shutil, requests

MODEL = "qwen2.5:3b"          # the local model Hermes will use as its brain (CPU-friendly)
OLLAMA_URL = "http://127.0.0.1:11434"

# 1) install Ollama if it isn't here yet
if shutil.which("ollama") is None:
    print("Installing Ollama ...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2) start the server on CPU (harmless if one is already running)
os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")
subprocess.Popen(["ollama", "serve"], env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        if requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok:
            break
    except Exception:
        time.sleep(1)

# 3) pull the model (first time downloads it)
print(f"Pulling {MODEL} (first time downloads weights) ...")
subprocess.run(["ollama", "pull", MODEL], check=True)
print(f"Ollama is ready with {MODEL}. This is the brain Hermes will use.")

## 2. Install the Hermes Agent

Hermes ships an official installer that sets up everything it needs (Python, Node.js, and a few tools). One command:

In [ ]:
import subprocess, os, shutil

print("Installing Hermes Agent (a few minutes — installs Python, Node.js, and tools) ...")
subprocess.run("curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash",
               shell=True, check=True)

# The installer puts the `hermes` command on your PATH for NEW shells. Add the usual
# locations to THIS notebook's PATH so the cells below can find it too.
for p in [os.path.expanduser("~/.local/bin"), os.path.expanduser("~/.hermes/bin")]:
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

hermes = shutil.which("hermes")
print("\nHermes installed." if hermes else
      "\nHermes installed (the `hermes` command will be available in a fresh Terminal).")
if hermes:
    print("hermes:", hermes)

## 3. Point Hermes at your local Ollama model

By default Hermes wants a cloud provider and an API key. We override that to use the **local Ollama** from Step 1 instead — free, on CPU, no key. Hermes reads its settings from two small files in `~/.hermes/`:

- **`config.yaml`** — which model/endpoint to use
- **`.env`** — secrets (we just put a *dummy* key, since a local endpoint doesn't check it)

This cell writes both for you.

In [ ]:
from pathlib import Path

hdir = Path.home() / ".hermes"
hdir.mkdir(parents=True, exist_ok=True)

# Use the local Ollama OpenAI-compatible endpoint as a "custom" provider.
(hdir / "config.yaml").write_text(
    "model:\n"
    "  provider: custom\n"
    "  base_url: http://localhost:11434/v1\n"
    f"  default: {MODEL}\n"
)
# A local endpoint ignores the key, but Hermes expects one to be set.
(hdir / ".env").write_text("OPENAI_API_KEY=dummy\n")

print("Configured Hermes to use local Ollama:")
print("  provider : custom (OpenAI-compatible)")
print("  endpoint : http://localhost:11434/v1")
print(f"  model    : {MODEL}")
print("\nWrote ~/.hermes/config.yaml and ~/.hermes/.env")

## 4. Quick check

Let's confirm the config landed and the `hermes` command is available.

In [ ]:
from pathlib import Path
import subprocess, shutil

print("=== ~/.hermes/config.yaml ===")
print((Path.home() / ".hermes" / "config.yaml").read_text())

hermes = shutil.which("hermes")
if hermes:
    try:
        out = subprocess.run([hermes, "--version"], capture_output=True, text=True, timeout=30)
        print("hermes --version:", (out.stdout or out.stderr).strip())
    except Exception as e:
        print("(couldn't run hermes --version here:", e, ")")
    print("\n✅ Ready. Now open a Terminal and run Hermes (next cell explains how).")
else:
    print("\nℹ️ The `hermes` command isn't on THIS notebook's PATH, but it will be in a fresh")
    print("   Terminal (the installer added it to your shell). Continue to the next step.")

## 5. 🚀 Now talk to your agent — in the Terminal

Hermes's chat is a full-screen terminal app, so we run it in the **Terminal**, not in this notebook.

**Open the Terminal** (in the Web Desktop) and run:

```bash
hermes
```

> If it says `command not found`, the installer added Hermes to your shell startup — open a **new** Terminal window, or run `source ~/.bashrc` first, then `hermes`.

The first time, Hermes may show a short setup screen — accept the defaults (it will use the local‑Ollama settings we wrote in Step 3).

### Try these — and watch what the *agent* does
Type these one at a time and watch the screen, not just the answer:

1. **`What is the NAIRR pilot, in two sentences?`**
   — a plain answer. This is the "engine" part: the model just responds.
2. **`What files are in my home directory? Use a tool to check, don't guess.`**
   — now watch Hermes **call a tool** (it runs a command on the machine) and answer from the *real* result. That tool‑use is the agent part.
3. **`Make a file called hello.txt that says "Hello from Hermes".`**
   — Hermes **takes an action** on your instance. Check it afterward with `ls` / `cat hello.txt`.

> 💡 **What to point out:** the model didn't just *describe* how to do these — Hermes turned the model's decision into **real actions** (running commands, writing files). That's the leap from an engine to an agent. With a small CPU model it won't be perfect at choosing tools every time — that's expected; the point is to *see the mechanism*.

To leave the chat, type `/exit` (or press `Ctrl+C`).

## 6. Recap & cleanup

**What you did:** installed a real AI **agent**, pointed it at a **free local model** on a CPU instance, and watched it **use tools and take actions** — not just generate text.

**Where to go next**
- `hermes model` — switch models or providers (e.g. a more capable cloud model with an API key, if you have one).
- `hermes tools` — see and configure what tools the agent is allowed to use.
- Hermes can also **remember** across sessions and **build its own skills** — explore the docs at <https://hermes-agent.nousresearch.com/docs>.

> 🧹 **Done for today?** Back in the Jetstream2 portal, **Shelve** or **Delete** your instance so it stops drawing allocation credits.